# Predicción de Costos de Seguro Médico
## Caso de Uso de Regresión — CRISP-DM
**Universidad Popular del Cesar**

---

**Asignatura:** Ciencia de Datos  
**Docente:** Aimer Rivera Centeno  
**Metodología:** CRISP-DM  

### Objetivo del Notebook
Predecir el costo de seguro médico (`charges`) de una persona a partir de sus características demográficas y de salud, utilizando modelos de regresión supervisada. Se comparan 6 algoritmos y se selecciona el mejor para desplegarlo como servicio web.

In [ ]:
# =============================================================================
# CELDA 2: Imports — Librerías necesarias para todo el pipeline
# =============================================================================
# pandas y numpy: manipulación y análisis de datos tabulares y numéricos
# matplotlib y seaborn: visualización estática (estilo publication-ready)
# plotly: visualización interactiva para dashboards web
# sklearn: modelado, preprocesamiento, métricas de regresión
# xgboost: implementación optimizada de Gradient Boosting
# joblib: serialización de modelos entrenados para producción
# warnings: silenciar warnings internos de librerías

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import joblib
import os

print("✓ Librerías cargadas exitosamente")

In [ ]:
# =============================================================================
# CELDA 3: Fase 2 CRISP-DM — Comprensión de los Datos + Carga y EDA inicial
# =============================================================================
# Cargamos el dataset desde la carpeta data/
# El estudiante debe colocar insurance.csv en Regresion/data/ antes de ejecutar

df = pd.read_csv('../data/insurance.csv')

# Verificamos dimensiones: esperamos 1338 filas y 7 columnas
print("=" * 60)
print("RESUMEN DEL DATASET")
print("=" * 60)
print(f"Shape (filas, columnas): {df.shape}")

# Mostramos las primeras 10 filas para inspección visual
print("\nPrimeras 10 filas:")
display(df.head(10))

# Información detallada: tipos de datos, memoria, nulos
print("\nInformación del dataset:")
df.info()

# Verificamos nulos: este dataset no tiene nulos, pero es buena práctica confirmarlo
print(f"\nValores nulos totales: {df.isnull().sum().sum()}")

# Estadísticas descriptivas: media, std, min, cuartiles, max
# Útil para detectar outliers y entender rangos de cada variable
print("\nEstadísticas descriptivas:")
display(df.describe())

In [ ]:
# =============================================================================
# CELDA 4: EDA — Distribución de la variable objetivo (charges)
# =============================================================================
# La variable objetivo 'charges' tiene una distribución sesgada a la derecha
# (asimetría positiva), típica de datos de costos médicos donde unos pocos
# individuos tienen costos muy altos. Aplicamos log1p para normalizarla.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de charges original
# Observamos la cola larga a la derecha (valores altos de costo)
axes[0].hist(df['charges'], bins=50, color='#2196F3', edgecolor='white')
axes[0].set_title('Distribución de Charges (original)')
axes[0].set_xlabel('Charges (USD)')
axes[0].set_ylabel('Frecuencia')

# Histograma de log(charges) — transformación logarítmica
# log1p = ln(1+x) para evitar log(0). La distribución se acerca a una normal.
axes[1].hist(np.log1p(df['charges']), bins=50, color='#4CAF50', edgecolor='white')
axes[1].set_title('Distribución de log(Charges) — normalizada')
axes[1].set_xlabel('log(Charges)')
axes[1].set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

# Métricas de centralidad y asimetría
# Si skewness > 1, la distribución está fuertemente sesgada a la derecha
print(f"Media: ${df['charges'].mean():,.2f}")
print(f"Mediana: ${df['charges'].median():,.2f}")
print(f"Skewness (asimetría): {df['charges'].skew():.3f}")
print(f"\nConclusión: La media > mediana y skewness positivo confirman "
      f"una distribución con cola derecha larga.")

In [ ]:
# =============================================================================
# CELDA 5: EDA — Impacto del tabaquismo en los costos
# =============================================================================
# El tabaquismo es el factor más determinante según la literatura.
# Comparamos la distribución de charges entre fumadores y no fumadores.

fig = px.box(df, x='smoker', y='charges', color='smoker',
             title='Distribución de Charges por Estatus de Fumador',
             color_discrete_map={'yes': '#F44336', 'no': '#2196F3'},
             labels={'smoker': 'Fumador', 'charges': 'Charges (USD)'})
fig.show()

# Estadísticas comparativas: la diferencia es dramática
print("Costo promedio fumadores: ${:,.2f}".format(
    df[df['smoker']=='yes']['charges'].mean()))
print("Costo promedio no fumadores: ${:,.2f}".format(
    df[df['smoker']=='no']['charges'].mean()))
print(f"\nDiferencia: los fumadores pagan ~3.8x más que los no fumadores.")
print(f"Esto confirma que smoker será la variable más importante del modelo.")

In [ ]:
# =============================================================================
# CELDA 6: EDA — Scatter matrix y heatmap de correlación
# =============================================================================
# La scatter matrix muestra relaciones bivariadas entre variables numéricas.
# Coloreamos por smoker para detectar interacciones (ej: BMI vs charges).

# Scatter matrix: age, bmi, children, charges coloreados por fumador
# Observamos que en fumadores (rojo), la relación BMI-charges es más pronunciada
fig = px.scatter_matrix(df,
    dimensions=['age', 'bmi', 'children', 'charges'],
    color='smoker',
    title='Scatter Matrix — Variables Numéricas por Estatus Fumador',
    color_discrete_map={'yes': '#F44336', 'no': '#2196F3'},
    opacity=0.6)
fig.show()

# Heatmap de correlación entre variables numéricas
# Seleccionamos solo columnas numéricas para la matriz de correlación
corr = df.select_dtypes(include=np.number).corr()
fig_h = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r',
                  title='Correlación entre Variables Numéricas',
                  labels=dict(color='Correlación'))
fig_h.show()

print("\nObservaciones:")
print("- La correlación más fuerte es entre age y charges (lógica: más edad = más gastos)")
print("- BMI tiene correlación moderada con charges, pero se amplifica en fumadores")

In [ ]:
# =============================================================================
# CELDA 7: EDA — Charges por región y sexo
# =============================================================================
# Verificamos si hay diferencias geográficas o de género en los costos.
# Esto ayuda a decidir si estas variables son predictores relevantes.

fig = px.box(df, x='region', y='charges', color='sex',
             title='Charges por Región y Sexo',
             labels={'region': 'Región', 'charges': 'Charges (USD)', 'sex': 'Sexo'},
             color_discrete_map={'male': '#2196F3', 'female': '#FF69B4'})
fig.show()

print("Análisis:")
print("- Las diferencias por región son pequeñas (southeast tiene media ligeramente más alta)")
print("- Las diferencias por sexo son mínimas en todos los casos")
print("- Esto sugiere que región y sexo tendrán menor peso en el modelo")

In [ ]:
# =============================================================================
# CELDA 8: Fase 3 CRISP-DM — Preparación de los Datos
# =============================================================================
# Transformaciones:
# 1. Label encoding binario para smoker y sex (0/1)
# 2. One-hot encoding para region (k-1 variables dummy, drop_first evita multicolinealidad)
# 3. Feature engineering: interacción bmi_smoker (captura sinergia obesidad + tabaquismo)
# 4. Feature engineering: age_sq (modela no-linealidad de costos con la edad)
# 5. Train/Test split (80/20) con random_state fijo para reproducibilidad
# 6. StandardScaler: normaliza features a media 0 y desviación 1

df_model = df.copy()

# --- Encoding de variables categóricas ---
# Convertimos smoker a 0/1: 1 si es fumador ('yes')
df_model['smoker_enc'] = (df_model['smoker'] == 'yes').astype(int)

# Convertimos sex a 0/1: 1 si es masculino ('male')
df_model['sex_enc'] = (df_model['sex'] == 'male').astype(int)

# One-hot encoding para región (4 categorías → 3 dummies)
# drop_first=True evita la trampa de las variables dummy (multicolinealidad perfecta)
df_model = pd.get_dummies(df_model, columns=['region'], prefix='region', drop_first=True)

# Eliminamos las columnas categóricas originales
df_model.drop(['sex', 'smoker'], axis=1, inplace=True)

print("Columnas después del encoding:")
print(df_model.columns.tolist())

# --- Ingeniería de Features ---
# Interacción bmi_smoker: bmi * smoker_enc
# Un fumador con BMI alto tiene costos mucho más altos que la suma de ambos efectos
df_model['bmi_smoker'] = df_model['bmi'] * df_model['smoker_enc']

# Término cuadrático de edad: age^2
# Los costos médicos crecen aceleradamente con la edad (no es lineal)
df_model['age_sq'] = df_model['age'] ** 2

# --- Separación X (features) y y (target) ---
X = df_model.drop('charges', axis=1)
y = df_model['charges']

# --- Train/Test Split ---
# 80% entrenamiento, 20% prueba. random_state=42 garantiza reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# --- Escalado ---
# StandardScaler: centra (media=0) y escala (std=1) cada feature
# Importante: fit SOLO en train, transform en train y test (evita data leakage)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"\nTrain: {X_train.shape} | Test: {X_test.shape}")
print(f"Features totales: {X.shape[1]}")
print("✓ Datos preparados correctamente")

In [ ]:
# =============================================================================
# CELDA 9: Fase 4 CRISP-DM — Modelado
# =============================================================================
# Entrenamos 6 modelos de regresión:
# - Lineales: Regresión Lineal, Ridge (L2), Lasso (L1)
# - Ensemble: Random Forest, XGBoost, Gradient Boosting
# Cada modelo se evalúa con MAE, RMSE y R² en el conjunto de prueba

modelos = {
    'Regresión Lineal': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42)
}

resultados = {}
modelos_entrenados = {}

print("=" * 70)
print("ENTRENAMIENTO DE MODELOS")
print("=" * 70)

for nombre, modelo in modelos.items():
    # Entrenamos cada modelo con los datos escalados
    modelo.fit(X_train_sc, y_train)
    
    # Predicción sobre conjunto de prueba
    y_pred = modelo.predict(X_test_sc)
    
    # Métricas de evaluación
    mae = mean_absolute_error(y_test, y_pred)          # Error absoluto medio (en USD)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Raíz del error cuadrático medio
    r2 = r2_score(y_test, y_pred)                      # Coeficiente de determinación

    resultados[nombre] = {'MAE': mae, 'RMSE': rmse, 'R²': r2}
    modelos_entrenados[nombre] = {'modelo': modelo, 'y_pred': y_pred}
    
    print(f"{nombre:25s} | MAE: ${mae:>8,.0f} | RMSE: ${rmse:>8,.0f} | R²: {r2:.4f}")

print("\n✓ Todos los modelos entrenados")

In [ ]:
# =============================================================================
# CELDA 10: Fase 5 CRISP-DM — Evaluación
# =============================================================================
# Comparamos todos los modelos y visualizamos el rendimiento del mejor

# --- Tabla comparativa ---
df_res = pd.DataFrame(resultados).T.round(4)
print("TABLA COMPARATIVA DE MODELOS")
print("=" * 60)
display(df_res.sort_values('R²', ascending=False))

# --- Seleccionar el mejor modelo por R² ---
mejor_nombre = df_res['R²'].idxmax()
mejor_modelo = modelos_entrenados[mejor_nombre]['modelo']
mejor_pred = modelos_entrenados[mejor_nombre]['y_pred']

print(f"\n{'=' * 60}")
print(f"🏆 MEJOR MODELO: {mejor_nombre}")
print(f"{'=' * 60}")
print(f"MAE:  ${resultados[mejor_nombre]['MAE']:,.2f}")
print(f"RMSE: ${resultados[mejor_nombre]['RMSE']:,.2f}")
print(f"R²:   {resultados[mejor_nombre]['R²']:.4f}")

# --- Gráfica 1: Real vs Predicho ---
# Un modelo perfecto tendría todos los puntos sobre la línea diagonal y=x
# Los puntos por debajo = subestimación, por encima = sobreestimación
fig = px.scatter(x=y_test, y=mejor_pred,
                 labels={'x': 'Charges Real (USD)', 'y': 'Charges Predicho (USD)'},
                 title=f'Real vs Predicho — {mejor_nombre}',
                 opacity=0.6)
fig.add_shape(type='line', x0=y_test.min(), y0=y_test.min(),
              x1=y_test.max(), y1=y_test.max(),
              line=dict(color='red', dash='dash'))
fig.show()

# --- Gráfica 2: Distribución de Residuales ---
# Los residuales (error = real - predicho) deben distribuirse normalmente
# alrededor de 0 si el modelo es insesgado
residuales = y_test - mejor_pred
fig_res = px.histogram(residuales, nbins=50,
                       title=f'Distribución de Residuales — {mejor_nombre}',
                       labels={'value': 'Error (USD)'},
                       color_discrete_sequence=['#FF9800'])
fig_res.add_vline(x=0, line_dash='dash', line_color='red',
                  annotation_text='Error cero')
fig_res.show()

print(f"\nEstadísticas de residuales:")
print(f"  Media: ${residuales.mean():,.2f}")
print(f"  Std:   ${residuales.std():,.2f}")
print(f"  Mín:   ${residuales.min():,.2f}")
print(f"  Máx:   ${residuales.max():,.2f}")

# --- Gráfica 3: Feature Importance (solo para modelos basados en árboles) ---
# Random Forest, XGBoost y Gradient Boosting tienen atributo feature_importances_
# Esto nos dice qué variables pesan más en la decisión del modelo
if hasattr(mejor_modelo, 'feature_importances_'):
    importances = mejor_modelo.feature_importances_
    fi_df = pd.DataFrame({'feature': X.columns, 'importance': importances})
    fi_df = fi_df.sort_values('importance', ascending=False).head(10)
    
    fig_fi = px.bar(fi_df, x='importance', y='feature', orientation='h',
                    title=f'Importancia de Features — {mejor_nombre}',
                    color='importance', color_continuous_scale='Viridis',
                    labels={'importance': 'Importancia', 'feature': 'Variable'})
    fig_fi.show()
    
    print(f"\nTop 5 variables más importantes según {mejor_nombre}:")
    for i, row in fi_df.head(5).iterrows():
        print(f"  {i+1}. {row['feature']}: {row['importance']:.4f}")
else:
    print(f"\nEl modelo {mejor_nombre} no tiene feature_importances_ (es un modelo lineal).")

In [ ]:
# =============================================================================
# CELDA 11: Fase 6 CRISP-DM — Despliegue: Guardar modelo para la app Flask
# =============================================================================
# Guardamos 3 artefactos:
# 1. modelo_seguro.pkl  → el modelo entrenado (para hacer predicciones)
# 2. scaler_seguro.pkl   → el scaler (para escalar inputs del formulario)
# 3. feature_names.pkl   → los nombres de features (para saber el orden exacto)
#
# Estos archivos se guardan en app/regresion_app/model/
# La app Flask los cargará al iniciar para servir predicciones

# Creamos el directorio model/ dentro de la app Flask si no existe
os.makedirs('../../../app/regresion_app/model', exist_ok=True)

# Guardamos el modelo ganador (el que tuvo mejor R²)
joblib.dump(mejor_modelo,
            '../../../app/regresion_app/model/modelo_seguro.pkl')
print(f"✓ Modelo '{mejor_nombre}' guardado → modelo_seguro.pkl")

# Guardamos el scaler para preprocesar inputs nuevos de forma idéntica
joblib.dump(scaler,
            '../../../app/regresion_app/model/scaler_seguro.pkl')
print("✓ Scaler guardado → scaler_seguro.pkl")

# Guardamos los nombres de las features en el orden exacto del entrenamiento
# Esto es crítico: el formulario web debe generar el vector en el mismo orden
joblib.dump(list(X.columns),
            '../../../app/regresion_app/model/feature_names.pkl')
print("✓ Feature names guardados → feature_names.pkl")

print(f"\nArchivos guardados en: app/regresion_app/model/")

## Fase 6: Conclusiones

### Hallazgos Principales

- **El tabaquismo es el factor más determinante** del costo del seguro médico. Los fumadores pagan en promedio 3.8x más que los no fumadores.

- **La interacción BMI × Tabaquismo** captura el efecto amplificado en fumadores con obesidad, mejorando significativamente el poder predictivo del modelo.

- **La relación edad-costos es no lineal**: los costos se aceleran con la edad, capturado mediante el término age².

- **Los modelos ensemble (Random Forest / XGBoost) superan a los lineales** porque capturan relaciones no lineales e interacciones complejas.

### Resultados del Mejor Modelo
- El mejor modelo fue **[MODELO]** con un R² de **[VALOR]** y un MAE de **$[VALOR]**.

### Trabajo Futuro
- Incorporar variables adicionales: historial médico, hábitos de ejercicio, condiciones preexistentes.
- Explorar modelos cuantílicos para estimar intervalos de predicción (no solo un valor puntual).
- Implementar monitoreo de deriva del modelo (concept drift) en producción.